# Live demo: the original Min-K% code, start to finish, in about a minute

**T4 GPU**, then run the cells one by one while recording. Uses Pythia-160M (smallest, fastest) on
200 WikiMIA passages of 64 words; the full sweep is `01_replication_sweep.ipynb`.

In [ ]:
# Clone the ORIGINAL implementation at the exact commit used throughout the project.
%cd -q /content
!git clone --quiet https://github.com/swj0419/detect-pretrain-code.git 2>/dev/null || echo 'already cloned'
!cd /content/detect-pretrain-code && git checkout --quiet a5048f93c4fa6491aab0638bb3e6fd5a662d7698 && git log -1 --format='original code @ %h (%ad)' --date=short
# run.py needs accelerate (device_map='auto') and imports openai unconditionally.
!pip install -q accelerate openai

import json, os, platform, sys
import torch, transformers, datasets, sklearn, numpy as np
ENV = {'python': platform.python_version(), 'torch': torch.__version__,
       'transformers': transformers.__version__, 'datasets': datasets.__version__,
       'sklearn': sklearn.__version__, 'numpy': np.__version__,
       'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}
print(json.dumps(ENV, indent=1))
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU, then Run all again.'

In [ ]:
# Import the original modules and call THEIR functions. Nothing in run.py / eval.py is edited;
# we only drive them from Python so the per-passage scores can be saved (run.py discards them).
SRC = '/content/detect-pretrain-code/src'
os.chdir(SRC); sys.path.insert(0, SRC)
import run as R            # original run.py  (load_model, evaluate_data, ...)
import gc, re, time
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

REF = 'EleutherAI/pythia-70m'   # the paper's reference model for Pythia-2.8B (Section 4.2)

def metric_name(key):
    """Map run.py's prediction keys to the paper's method names."""
    if key == 'ppl': return 'PPL'
    if key.startswith('ppl/Ref'): return 'Smaller Ref'
    if key.startswith('ppl/lowercase'): return 'Lowercase'
    if key.startswith('ppl/zlib'): return 'Zlib'
    m = re.match(r'Min_([0-9.]+)% Prob', key)          # note: run.py writes 'Min_30.000000000000004% Prob'
    return f'Min-{round(float(m.group(1)))}%' if m else key

BOOT_METRICS = {'PPL', 'Zlib', 'Lowercase', 'Smaller Ref', 'Min-20%'}

def summarise(outputs, model, dataset, length, n_boot=1000, seed=0):
    """AUC + TPR@5%FPR per method using the ORIGINAL eval.sweep, plus a bootstrap 95% CI on AUC."""
    labels = np.array([ex['label'] for ex in outputs], dtype=bool)
    rng = np.random.default_rng(seed)
    idx = [rng.integers(0, len(labels), len(labels)) for _ in range(n_boot)]
    rows = []
    for key in outputs[0]['pred']:
        scores = np.array([ex['pred'][key] for ex in outputs], dtype=float)
        if np.isnan(scores).any():
            continue
        fpr, tpr, auc, acc = R.sweep(scores, labels)          # original: roc_curve(labels, -score)
        tpr5 = tpr[np.where(fpr < .05)[0][-1]]
        lo = hi = np.nan
        name = metric_name(key)
        if name in BOOT_METRICS:
            boots = [roc_auc_score(labels[i], -scores[i]) for i in idx if 0 < labels[i].sum() < len(i)]
            lo, hi = np.percentile(boots, [2.5, 97.5])
        rows.append(dict(dataset=dataset, model=model, length=length, n=len(labels), method=name,
                         auc=auc, auc_lo=lo, auc_hi=hi, tpr_at_5fpr=tpr5, best_acc=acc))
    return rows

def save_jsonl(rows, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as f:
        for r in rows:
            f.write(json.dumps({'input': r['input'], 'label': int(r['label']), 'pred': r['pred']}) + '\n')

def run_model(target, jobs, out_root, dataset_name):
    """jobs: list of (length, list_of_examples). Loads the model once, scores every job."""
    rows = []
    todo = [(L, data) for L, data in jobs
            if not os.path.exists(f'{out_root}/scores/{target.split("/")[-1]}_len{L}.jsonl')]
    for L, data in jobs:                                  # resume: reuse finished runs
        p = f'{out_root}/scores/{target.split("/")[-1]}_len{L}.jsonl'
        if os.path.exists(p):
            done = [json.loads(l) for l in open(p)]
            rows += summarise(done, target, dataset_name, L)
    if not todo:
        return rows
    t0 = time.time()
    m1, m2, t1, t2 = R.load_model(target, REF)             # original loader
    print(f'{target}: loaded in {time.time()-t0:.0f}s, dtype={m1.dtype}')
    for L, data in todo:
        t0 = time.time()
        out = R.evaluate_data(data, m1, m2, t1, t2, 'input', target, REF)   # original scoring loop
        d = f'{out_root}/original_eval/{target}_{REF}/len{L}'
        os.makedirs(d, exist_ok=True)
        try:
            R.fig_fpr_tpr(out, d)                          # original auc.txt + auc.png
        except ValueError as e:                            # e.g. NaN from Min-5% on a very short passage
            print('  original eval.py failed:', e)
        plt.close('all')
        save_jsonl(out, f'{out_root}/scores/{target.split("/")[-1]}_len{L}.jsonl')
        rows += summarise(out, target, dataset_name, L)
        print(f'  length {L}: {len(data)} passages in {time.time()-t0:.0f}s')
    del m1, m2; gc.collect(); torch.cuda.empty_cache()
    return rows
print('driver ready')

## 0. Load the model and data (run this BEFORE you start recording)

In [ ]:
TARGET = 'EleutherAI/pythia-160m'
m1, m2, t1, t2 = R.load_model(TARGET, REF)                       # original loader
data = R.convert_huggingface_data_to_list_dic(R.load_dataset('swj0419/WikiMIA', split='WikiMIA_length64'))
print('ready:', TARGET, '+', REF, '|', len(data), 'WikiMIA passages of 64 words')

## 1. Score passages with the original `run.py` (live, ~10 s)

In [ ]:
out = R.evaluate_data(data[:200], m1, m2, t1, t2, 'input', TARGET, REF)   # original scoring loop

## 2. What Min-K% looks at: the least likely 20% of tokens in one passage

In [ ]:
def show(ex):
    _, logp, _ = R.calculatePerplexity(ex['input'], m1, t1, m1.device)     # original per-token log-probs
    toks = t1.convert_ids_to_tokens(t1.encode(ex['input']))[1:]
    k = int(len(logp) * 0.2)
    worst = sorted(zip(logp, toks))[:k]
    kind = 'MEMBER (pre-2017)' if ex['label'] == 1 else 'NON-MEMBER (post-2023)'
    print(f'{kind}: {ex["input"][:110]}...')
    print('  lowest-20% tokens:', ', '.join(f'{t.replace("Ġ", "")}({p:.1f})' for p, t in worst[:8]), '...')
    print(f'  Min-20% score = {np.mean([p for p, _ in worst]):.2f}   (higher = more likely a member)\n')

show(next(ex for ex in out if ex['label'] == 1))
show(next(ex for ex in out if ex['label'] == 0))

## 3. The original evaluation: `eval.py` writes `auc.txt` and the ROC plot

In [ ]:
os.makedirs('/content/demo', exist_ok=True)
R.fig_fpr_tpr(out, '/content/demo')                              # original evaluation
print(open('/content/demo/auc.txt').read())
from IPython.display import Image, display
display(Image('/content/demo/auc.png'))